In [1]:
import pandas as pd

# Mark 1

In [2]:
import pandas as pd

def parse_section(linhas, section_name):
    """Extrai as linhas de uma seção específica sem processar o conteúdo."""
    data = []
    coletando = False
    for linha in linhas:
        linha_limpa = linha.strip()
        if linha_limpa.upper().startswith(f"[{section_name.upper()}]"):
            coletando = True
            continue
        if coletando:
            if linha_limpa.startswith("["):
                break
            if linha_limpa and not linha_limpa.startswith(";"):
                data.append(linha_limpa.split())
    return data

def merge_inps(arq1, arq2, output_path):
    # Carregar linhas de ambos
    with open(arq1, 'r', encoding='latin1') as f:
        linhas1 = f.readlines()
    with open(arq2, 'r', encoding='latin1') as f:
        linhas2 = f.readlines()

    # Seções que representam NÓS (prefixo N)
    secoes_nodes = ["JUNCTIONS", "RESERVOIRS", "TANKS", "COORDINATES"]
    # Seções que representam LINKS (prefixo T e referenciam NÓS)
    secoes_links = ["PIPES", "PUMPS", "VALVES"]
    # Outras seções que usam links (como vértices)
    secoes_extras = ["VERTICES", "STATUS", "CONTROLS"]

    # Dicionário para armazenar o conteúdo final
    merged_data = {}

    # 1. Pegar todas as seções do arquivo 1 como base
    todas_secoes = ["JUNCTIONS", "RESERVOIRS", "TANKS", "PIPES", "PUMPS", "VALVES", "COORDINATES", "VERTICES"]
    
    for sec in todas_secoes:
        merged_data[sec] = parse_section(linhas1, sec)

    # 2. Processar arquivo 2 com prefixos e adicionar ao merged_data
    for sec in todas_secoes:
        data2 = parse_section(linhas2, sec)
        
        for row in data2:
            if sec in secoes_nodes:
                # Ex: [ID, Elev, Demand...] -> ID vira N + ID
                row[0] = "N" + row[0]
            
            elif sec in secoes_links:
                # Ex Pipes: [ID, Node1, Node2...] -> ID vira T, Nodes viram N
                row[0] = "T" + row[0]
                row[1] = "N" + row[1]
                row[2] = "N" + row[2]
            
            elif sec == "VERTICES":
                # [LinkID, X, Y] -> LinkID vira T
                row[0] = "T" + row[0]
        
        merged_data[sec].extend(data2)

    # 3. Montar o novo arquivo .inp
    with open(output_path, 'w', encoding='utf-8') as f:
        # Aqui você pode copiar o cabeçalho do arquivo 1 (Options, Times, etc.)
        # Para simplificar, vamos focar nas seções de dados:
        
        for sec, rows in merged_data.items():
            if rows:
                f.write(f"[{sec}]\n")
                if sec == "JUNCTIONS":
                    f.write(";ID               Elev          Demand\n")
                elif sec == "PIPES":
                    f.write(";ID               Node1            Node2            Length\n")
                
                for r in rows:
                    f.write("\t".join(r) + "\n")
                f.write("\n")

    print(f"Sucesso! Arquivo fundido salvo em: {output_path}")

# Exemplo de uso:
# merge_inps("rede_principal.inp", "expansao.inp", "modelo_final.inp")

In [3]:
merge_inps("Fusão\\Criacao\\NBN.inp", "Fusão\\Criacao\\SSB.inp", "Fusão\\Criacao\\Fusao.inp")

Sucesso! Arquivo fundido salvo em: Fusão\Criacao\Fusao.inp


# Mark 2

In [2]:
import pandas as pd

def ler_secoes_completas(linhas):
    """Lê TODAS as seções do arquivo mantendo estrutura original"""
    secoes = {}
    sec_atual = None

    for linha in linhas:
        linha_strip = linha.strip()

        if linha_strip.startswith("[") and linha_strip.endswith("]"):
            sec_atual = linha_strip.replace("[", "").replace("]", "")
            secoes[sec_atual] = []
            continue

        if sec_atual:
            secoes[sec_atual].append(linha.rstrip("\n"))

    return secoes


def parse_section(linhas, section_name):
    data = []
    coletando = False

    for linha in linhas:
        linha_limpa = linha.strip()

        if linha_limpa.upper().startswith(f"[{section_name.upper()}]"):
            coletando = True
            continue

        if coletando:
            if linha_limpa.startswith("["):
                break
            if linha_limpa and not linha_limpa.startswith(";"):
                data.append(linha_limpa.split())

    return data


def merge_inps(arq1, arq2, output_path):

    with open(arq1, 'r', encoding='latin1') as f:
        linhas1 = f.readlines()

    with open(arq2, 'r', encoding='latin1') as f:
        linhas2 = f.readlines()

    # 🔹 Ler TODAS as seções do arquivo 1
    secoes_final = ler_secoes_completas(linhas1)

    # 🔹 Seções que vamos tratar
    secoes_nodes = ["JUNCTIONS"]
    secoes_links = ["PIPES"]
    secoes_coords = ["COORDINATES"]
    secoes_vertices = ["VERTICES"]

    secoes_tratadas = secoes_nodes + secoes_links + secoes_coords + secoes_vertices

    # 🔹 Garantir que essas seções existam
    for sec in secoes_tratadas:
        if sec not in secoes_final:
            secoes_final[sec] = []

    # 🔹 Processar arquivo 2
    for sec in secoes_tratadas:
        data2 = parse_section(linhas2, sec)

        for row in data2:

            if sec in secoes_nodes:
                row[0] = "N" + row[0]

            elif sec in secoes_links:
                row[0] = "T" + row[0]
                row[1] = "N" + row[1]
                row[2] = "N" + row[2]

            elif sec in secoes_coords:
                row[0] = "N" + row[0]

            elif sec in secoes_vertices:
                row[0] = "T" + row[0]

        # Converter para string formatada
        linhas_formatadas = ["\t".join(r) for r in data2]

        # Adicionar ao final da seção existente
        secoes_final[sec].extend(linhas_formatadas)

    # 🔹 VALIDAÇÃO
    print("\n🔎 Validando duplicidades...")

    def extrair_ids(sec):
        return [
            linha.split()[0]
            for linha in secoes_final.get(sec, [])
            if linha and not linha.startswith(";")
        ]

    def verificar_duplicados(ids, tipo):
        duplicados = set([x for x in ids if ids.count(x) > 1])
        if duplicados:
            print(f"❌ IDs duplicados em {tipo}: {duplicados}")
        else:
            print(f"✅ Sem duplicidade em {tipo}")

    verificar_duplicados(extrair_ids("JUNCTIONS"), "JUNCTIONS")
    verificar_duplicados(extrair_ids("PIPES"), "PIPES")

    # 🔹 Escrever arquivo final
    with open(output_path, 'w', encoding='utf-8') as f:

        for sec, linhas in secoes_final.items():
            if linhas:
                f.write(f"[{sec}]\n")

                for linha in linhas:
                    f.write(linha + "\n")

                f.write("\n")

    print(f"\n✅ Arquivo final gerado: {output_path}")

In [5]:
merge_inps(
    "Fusão\\Samambaia\\NOVAS VRPS.inp",
    "Fusão\\Samambaia\\QSA_V0.inp",
    "Fusão\\Samambaia\\Samambaia+QSA (NVRPs).inp")


🔎 Validando duplicidades...
✅ Sem duplicidade em JUNCTIONS
✅ Sem duplicidade em PIPES

✅ Arquivo final gerado: Fusão\Samambaia\Samambaia+QSA (NVRPs).inp
